# Data Preparation Pipeline

## Objective

Prepare the dataset for machine learning while preserving the integrity of the original raw dataset.

The preprocessing workflow is designed to be reproducible, modular, and suitable for production deployment.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
working_df = df.copy()

# Data Preparation Workflow

The preprocessing pipeline follows the order below:

1. Create Working Copy
2. Remove Identifier Columns
3. Fix Data Types
4. Handle Missing Values
5. Validate Dataset
6. Encode Features
7. Scale Numerical Features
8. Train/Test Split
9. Save Processed Dataset

In [3]:
print("=" * 60)
print("DATA PREPARATION INITIALIZATION")
print("=" * 60)

print(f"Rows    : {working_df.shape[0]}")
print(f"Columns : {working_df.shape[1]}")

print("\nWorking copy successfully created.")

DATA PREPARATION INITIALIZATION
Rows    : 7043
Columns : 21

Working copy successfully created.


In [4]:
print("Before Processing:")
print(working_df.shape)

Before Processing:
(7043, 21)


In [5]:
import numpy as np

working_df["TotalCharges"] = (
    working_df["TotalCharges"]
    .replace(r"^\s*$", np.nan, regex=True)
)

In [6]:
working_df["TotalCharges"].isnull().sum()

np.int64(11)

In [7]:
working_df[
    working_df["TotalCharges"].isnull()
][
    ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [8]:
working_df["TotalCharges"] = pd.to_numeric(
    working_df["TotalCharges"],
    errors="coerce"
)

working_df["TotalCharges"] = working_df["TotalCharges"].fillna(0)

In [9]:
print("Missing TotalCharges:", working_df["TotalCharges"].isnull().sum())
print("Dataset shape:", working_df.shape)

Missing TotalCharges: 0
Dataset shape: (7043, 21)


### Missing Value Handling — TotalCharges

The `TotalCharges` column contained 11 whitespace values that were converted to missing values.

All 11 affected customers had a tenure of 0 months, indicating newly acquired customers with no accumulated charges. Therefore, `TotalCharges` was imputed with 0 rather than removing these valid records.

This preserves the complete dataset while maintaining a business-consistent interpretation of the missing values.

In [10]:
print("=" * 60)
print("MISSING VALUE VERIFICATION")
print("=" * 60)

print("Total missing values:", working_df.isnull().sum().sum())
print("Dataset shape:", working_df.shape)

MISSING VALUE VERIFICATION
Total missing values: 0
Dataset shape: (7043, 21)


# Outlier Handling

Outliers are identified in numerical features using the Interquartile Range (IQR) method. Potentially extreme values are retained when they represent valid customer behavior rather than data errors.

In [11]:
print("=" * 60)
print("OUTLIER ANALYSIS")
print("=" * 60)

numerical_columns_52 = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

outlier_summary_52 = []

for column in numerical_columns_52:
    q1 = working_df[column].quantile(0.25)
    q3 = working_df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outlier_count = ((working_df[column] < lower_bound) | (working_df[column] > upper_bound)).sum()

    outlier_summary_52.append({
        "Feature": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": int(outlier_count)
    })

outlier_summary_52 = pd.DataFrame(outlier_summary_52)
outlier_summary_52

OUTLIER ANALYSIS


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count
0,tenure,9.00,55.00,46.00,-60.000,124.000,0
1,MonthlyCharges,35.50,89.85,54.35,-46.025,171.375,0
2,TotalCharges,398.55,3786.60,3388.05,-4683.525,8868.675,0


## Outlier Handling Decision

Potential IQR-based outliers are retained because extreme values in `tenure`, `MonthlyCharges`, and `TotalCharges` can represent valid customer behavior rather than erroneous observations.

Therefore, no rows are removed and no values are clipped during this step. The numerical features remain unchanged for subsequent preprocessing.

In [12]:
print("=" * 60)
print("OUTLIER HANDLING VERIFICATION")
print("=" * 60)
            
print("Dataset shape:", working_df.shape)
print("Total missing values:", working_df.isnull().sum().sum())
print("Outlier treatment: No rows removed and no values clipped.")

OUTLIER HANDLING VERIFICATION
Dataset shape: (7043, 21)
Total missing values: 0
Outlier treatment: No rows removed and no values clipped.


# Categorical Feature Encoding

Categorical features are converted into numerical representations suitable for machine learning.

- Identifier columns are excluded from model features.
- Categorical features are encoded using One-Hot Encoding.
- The target variable is encoded separately.
- The original raw dataset remains unchanged.

In [14]:
print("=" * 60)
print("CATEGORICAL FEATURE ENCODING")
print("=" * 60)

working_df_53 = working_df.drop(columns=["customerID"]).copy()

# Encode target separately
working_df_53["Churn"] = working_df_53["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Identify categorical input features before encoding
categorical_columns_53 = working_df_53.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# One-hot encode categorical input features
working_df_53 = pd.get_dummies(
    working_df_53,
    columns=categorical_columns_53,
    drop_first=True,
    dtype=int
)

print("Categorical columns encoded:", len(categorical_columns_53))
print("Encoded dataset shape:", working_df_53.shape)

CATEGORICAL FEATURE ENCODING
Categorical columns encoded: 15
Encoded dataset shape: (7043, 31)


In [15]:
print("=" * 60)
print("ENCODING VERIFICATION")
print("=" * 60)

remaining_categorical_53 = working_df_53.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Remaining categorical columns:", remaining_categorical_53)
print("Target unique values:", sorted(working_df_53["Churn"].unique()))
print("Dataset shape:", working_df_53.shape)

ENCODING VERIFICATION
Remaining categorical columns: []
Target unique values: [np.int64(0), np.int64(1)]
Dataset shape: (7043, 31)


## Encoding Decision

Categorical input features are encoded using One-Hot Encoding with the first category dropped to avoid redundant dummy variables.

The target variable `Churn` is encoded separately:

- `No` → 0
- `Yes` → 1

The identifier column `customerID` is excluded from model features.

The original raw dataset remains unchanged, while `working_df_53` contains the encoded representation for subsequent preprocessing steps.

# Feature Scaling

Numerical features are prepared for standardization using `StandardScaler`.

Only continuous numerical features are selected for scaling:

- `tenure`
- `MonthlyCharges`
- `TotalCharges`

One-hot encoded categorical features are kept unchanged.

The scaler is configured at this stage but will be fitted only on training data during the final preprocessing pipeline to prevent data leakage.

In [16]:
from sklearn.preprocessing import StandardScaler

numerical_columns_54 = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

scaler_54 = StandardScaler()

print("=" * 60)
print("FEATURE SCALING CONFIGURATION")
print("=" * 60)

print("Scaling method:", "StandardScaler")
print("Features to scale:", numerical_columns_54)
print("Features excluded from scaling: One-hot encoded features")

FEATURE SCALING CONFIGURATION
Scaling method: StandardScaler
Features to scale: ['tenure', 'MonthlyCharges', 'TotalCharges']
Features excluded from scaling: One-hot encoded features


In [17]:
print("=" * 60)
print("SCALING VERIFICATION")
print("=" * 60)

print("Scaler:", type(scaler_54).__name__)
print("Number of features selected:", len(numerical_columns_54))
print("Selected features:", numerical_columns_54)

SCALING VERIFICATION
Scaler: StandardScaler
Number of features selected: 3
Selected features: ['tenure', 'MonthlyCharges', 'TotalCharges']


## Scaling Decision

`StandardScaler` is selected because the numerical features have different ranges and scales.

The scaler is intentionally not fitted on the complete dataset at this stage. It will be fitted only on the training data to prevent information leakage.

One-hot encoded categorical features remain unscaled.

# Train-Test Split

The encoded dataset is divided into training and testing subsets before model development.

A stratified split is used to preserve the original churn class distribution in both subsets.

- Training set: 80%
- Testing set: 20%
- Random state: 42
- Stratification: `Churn`

The split is performed before fitting any data-dependent preprocessing or machine learning model to prevent data leakage.

In [18]:
from sklearn.model_selection import train_test_split

X_55 = working_df_53.drop(columns=["Churn"])
y_55 = working_df_53["Churn"]

X_train_55, X_test_55, y_train_55, y_test_55 = train_test_split(
    X_55,
    y_55,
    test_size=0.20,
    random_state=42,
    stratify=y_55
)

print("=" * 60)
print("TRAIN-TEST SPLIT")
print("=" * 60)

print("Training features:", X_train_55.shape)
print("Testing features:", X_test_55.shape)
print("Training target:", y_train_55.shape)
print("Testing target:", y_test_55.shape)

TRAIN-TEST SPLIT
Training features: (5634, 30)
Testing features: (1409, 30)
Training target: (5634,)
Testing target: (1409,)


In [19]:
print("=" * 60)
print("SPLIT VERIFICATION")
print("=" * 60)

print("Training churn distribution:")
print(y_train_55.value_counts(normalize=True).round(3))

print("\nTesting churn distribution:")
print(y_test_55.value_counts(normalize=True).round(3))

print("\nTotal records:", len(X_train_55) + len(X_test_55))

SPLIT VERIFICATION
Training churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Testing churn distribution:
Churn
0    0.735
1    0.265
Name: proportion, dtype: float64

Total records: 7043


# Build Preprocessing Pipeline

A reusable preprocessing pipeline is created to combine categorical encoding and numerical feature scaling into a single transformation workflow.

The pipeline is fitted only on the training data to prevent data leakage.

The identifier column `customerID` is excluded, while the target variable `Churn` remains separate from the preprocessing features.

In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X_56 = working_df.drop(columns=["customerID", "Churn"]).copy()

y_56 = working_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

categorical_columns_56 = X_56.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_columns_56 = X_56.select_dtypes(
    include=["number"]
).columns.tolist()

preprocessor_56 = ColumnTransformer(
    transformers=[
        (
            "numerical",
            StandardScaler(),
            numerical_columns_56
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_columns_56
        )
    ]
)

print("=" * 60)
print("PREPROCESSING PIPELINE")
print("=" * 60)

print("Numerical features:", numerical_columns_56)
print("Categorical features:", categorical_columns_56)

PREPROCESSING PIPELINE
Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [24]:
X_train_56, X_test_56, y_train_56, y_test_56 = train_test_split(
    X_56,
    y_56,
    test_size=0.20,
    random_state=42,
    stratify=y_56
)

X_train_processed_56 = preprocessor_56.fit_transform(X_train_56)
X_test_processed_56 = preprocessor_56.transform(X_test_56)

print("=" * 60)
print("PIPELINE VERIFICATION")
print("=" * 60)

print("Training data:", X_train_processed_56.shape)
print("Testing data:", X_test_processed_56.shape)
print("Training target:", y_train_56.shape)
print("Testing target:", y_test_56.shape)

PIPELINE VERIFICATION
Training data: (5634, 30)
Testing data: (1409, 30)
Training target: (5634,)
Testing target: (1409,)


## Preprocessing Pipeline Decision

The final preprocessing workflow uses `ColumnTransformer` to apply:

- `StandardScaler` to numerical features.
- `OneHotEncoder` to categorical features.
- `handle_unknown="ignore"` to safely process unseen categories during inference.

The preprocessing pipeline is fitted only on training data and then applied to the test data.

This prevents preprocessing-related data leakage and provides a reusable transformation workflow for subsequent model development and deployment.

In [25]:
print("=" * 60)
print("FINAL PREPROCESSING VERIFICATION")
print("=" * 60)

print("Training rows:", X_train_56.shape[0])
print("Testing rows:", X_test_56.shape[0])
print("Processed training features:", X_train_processed_56.shape[1])
print("Processed testing features:", X_test_processed_56.shape[1])
print("Total records:", X_train_56.shape[0] + X_test_56.shape[0])

FINAL PREPROCESSING VERIFICATION
Training rows: 5634
Testing rows: 1409
Processed training features: 30
Processed testing features: 30
Total records: 7043


# Feature Selection

Feature selection is performed to identify and remove non-informative features from the processed feature set.

The selection strategy focuses on zero-variance features, while preserving potentially useful predictive features for later model-based validation.

Model-based feature importance and SHAP analysis will be used later to evaluate predictive relevance.

In [26]:
from sklearn.feature_selection import VarianceThreshold

feature_selector_61 = VarianceThreshold(threshold=0)

feature_selector_61.fit(X_train_processed_56)

selected_feature_mask_61 = feature_selector_61.get_support()

feature_names_61 = preprocessor_56.get_feature_names_out()

selected_feature_names_61 = feature_names_61[selected_feature_mask_61]
removed_feature_names_61 = feature_names_61[~selected_feature_mask_61]

print("=" * 60)
print("FEATURE SELECTION")
print("=" * 60)

print("Features before selection:", len(feature_names_61))
print("Features after selection:", len(selected_feature_names_61))
print("Features removed:", len(removed_feature_names_61))

FEATURE SELECTION
Features before selection: 30
Features after selection: 30
Features removed: 0


In [27]:
print("=" * 60)
print("FEATURE SELECTION VERIFICATION")
print("=" * 60)

if len(removed_feature_names_61) == 0:
    print("No zero-variance features identified.")
else:
    print("Removed features:")
    for feature in removed_feature_names_61:
        print("-", feature)

print("Selected feature count:", len(selected_feature_names_61))

FEATURE SELECTION VERIFICATION
No zero-variance features identified.
Selected feature count: 30


## Feature Selection Decision

No potentially useful features are removed solely based on their predictive importance at this stage.

Zero-variance features, if identified, are removed because they provide no information to the model.

The remaining features are retained for model development, where model-based feature importance and SHAP explainability will provide further evidence of predictive relevance.